## 05_gravity_network
  Builds the observed CZ NUTS3 commuting matrix (14 regions, 2021 census) and fits singly
  constrained gravity (power-law, exponential) and radiation models with scikit-mobility.
  Output: data/processed/05_gravity_cz/ (observed OD, parameters, metrics, realizations), graphml.
  Environment: nuts3-mobility.

In [ ]:
# PROJ data fix: must run BEFORE geopandas / pyproj / skmob are imported
import os
import sys
from pathlib import Path
from importlib.util import find_spec


def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec and spec.submodule_search_locations:  # pip wheel with bundled data
        pyproj_dir = Path(list(spec.submodule_search_locations)[0])
        candidates.append(pyproj_dir / "proj_dir" / "share" / "proj")
    for c in candidates:
        if (c / "proj.db").exists():
            return c
    return None


proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)
    print("PROJ data: proj.db found, environment variables set")
else:
    print("PROJ data: proj.db NOT found in the expected places")

import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import requests
import shapely.wkb
from geopandas.array import GeometryArray
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import skmob
from skmob.models.gravity import Gravity
from skmob.models.radiation import Radiation

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
COUNTRY = "CZ"
YEAR = 2023                      # same year as notebook 04
CAPITAL_ID = "CZ010"             # Praha (set explicitly; the largest region is NOT the capital)
N_REPS = 20                      # stochastic realizations per model
SEED_BASE = 0                    # realization i uses seed SEED_BASE + i
CRS_GEO = "EPSG:4326"
CRS_METRIC = "EPSG:3035"

# own table columns
COL_ID, COL_POP = "nuts3_id", "population"
COL_COUNTRY, COL_YEAR = "country_code", "year"

# commuting source (original Czech column names stay unchanged)
CSV_URL = ("https://csu.gov.cz/docs/107508/8dc6b1a9-6932-6b27-7696-78a2f4d789e8/"
           "sldb2021_vyjizdka_pracoviste_kraj_pohlavi.csv?version=1.0")
CSV_NAME = "sldb2021_vyjizdka_pracoviste_kraj_pohlavi.csv"
COL_ORIGIN_LEVEL, COL_DEST_LEVEL = "uzemi_cis", "kraj_dojizdky_cis"
COL_DEST_CODE, COL_SEX = "kraj_dojizdky_kod", "pohlavi_kod"
COL_ORIGIN_NAME, COL_DEST_NAME, COL_VALUE = "uzemi_txt", "kraj_dojizdky_txt", "hodnota"
KRAJ_LEVEL_CODE = 100            # code of the kraj level in the code list

# inputs / outputs (relative paths only)
NUTS3_TABLE = config.PROCESSED_DIR / "nuts3_master_table.parquet"
GISCO_GEOJSON = config.RAW_DIR / "nuts3" / "NUTS_RG_03M_2024_4326.geojson"
RAW_COMMUTE_DIR = config.RAW_DIR / "commuting" / "cz"
CHECKPOINT_DIR = config.PROCESSED_DIR / "05_gravity_cz"   # realizations + summary tables
FIG_DIRS = [Path("../figures"), Path("../plots")]
GRAPH_PATH = config.PROCESSED_DIR / "graph_cz_gravity.graphml"
MODELS = ["gravity_power", "gravity_exp", "radiation"]
# ------------------------------------------------------------------

for d in [RAW_COMMUTE_DIR, CHECKPOINT_DIR, *FIG_DIRS]:
    d.mkdir(parents=True, exist_ok=True)

print("Config OK:", COUNTRY, YEAR, f"{N_REPS} realizations/model")

In [ ]:
# Load the master table; geometry is stored as WKB bytes
nuts3 = pd.read_parquet(NUTS3_TABLE)

# Element-wise geometry assignment (vectorized assignment fails with this shapely version)
geoms = nuts3["geometry"].apply(
    lambda x: shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x
).tolist()
geom_arr = np.empty(len(geoms), dtype=object)
for i, g in enumerate(geoms):
    geom_arr[i] = g

nuts3 = nuts3.drop(columns="geometry")
nuts3 = gpd.GeoDataFrame(nuts3, geometry=GeometryArray(geom_arr, crs=CRS_GEO))

# Select the country and year
cz = (nuts3[(nuts3[COL_COUNTRY] == COUNTRY) & (nuts3[COL_YEAR] == YEAR)]
      .copy().reset_index(drop=True))

# Centroids and pairwise distances in a metric projection
cz_metric = cz.to_crs(CRS_METRIC)
coords = np.array([[p.x, p.y] for p in cz_metric.geometry.centroid])
dist_matrix_km = squareform(pdist(coords)) / 1000   # metres -> km

# Capital comes from the config; check that it exists in the loaded regions
assert CAPITAL_ID in set(cz[COL_ID]), "CAPITAL_ID not found among the regions"
capital_pop_share = cz.loc[cz[COL_ID] == CAPITAL_ID, COL_POP].iloc[0] / cz[COL_POP].sum()

n_regions = len(cz)
print("Regions:", n_regions)
print("Distance matrix:", dist_matrix_km.shape)
print("Capital:", CAPITAL_ID, "| population share:", round(capital_pop_share, 3))
print("Missing population values:", int(cz[COL_POP].isna().sum()))

In [ ]:
import os

csv_path = RAW_COMMUTE_DIR / CSV_NAME
csv_rel = os.path.relpath(csv_path)   # relative path only, safe to print

if csv_path.exists():
    print("CSV already present, download skipped:", csv_rel)
else:
    try:
        response = requests.get(CSV_URL, timeout=120)
        response.raise_for_status()
        csv_path.write_bytes(response.content)
        print("Downloaded:", csv_rel)
    except Exception as e:
        print("Download failed:", type(e).__name__)
        print("Manual fix: open CSV_URL (config cell) in a browser and save the file as")
        print("  ", CSV_NAME, "into the folder:", os.path.relpath(RAW_COMMUTE_DIR))
        raise FileNotFoundError(CSV_NAME) from None

print("Size (MB):", round(csv_path.stat().st_size / 1e6, 2))

cz_commute_raw = pd.read_csv(csv_path)
print("Rows, columns:", cz_commute_raw.shape)
cz_commute_raw.head()

In [ ]:
# Local config: columns used only for checking (original Czech names)
COL_ACTIVITY, COL_INDICATOR = "aktivita_txt", "ukaz_txt"

# Keep only kraj -> kraj rows with both sexes combined (original Czech names unchanged)
mask = (
    (cz_commute_raw[COL_ORIGIN_LEVEL] == KRAJ_LEVEL_CODE)
    & (cz_commute_raw[COL_DEST_LEVEL] == KRAJ_LEVEL_CODE)
    & cz_commute_raw[COL_DEST_CODE].notna()
    & cz_commute_raw[COL_SEX].isna()
)
cz_kk = cz_commute_raw.loc[mask].copy()

# Each origin-destination pair must appear exactly once before pivoting
dup_pairs = int(cz_kk.duplicated(subset=[COL_ORIGIN_NAME, COL_DEST_NAME]).sum())

od_by_name = cz_kk.pivot(index=COL_ORIGIN_NAME, columns=COL_DEST_NAME, values=COL_VALUE)
n_missing_cells = int(od_by_name.isna().sum().sum())
od_by_name = od_by_name.fillna(0)

same_names = set(od_by_name.index) == set(od_by_name.columns)
if same_names:
    od_by_name = od_by_name.reindex(columns=od_by_name.index)  # same order on both axes
    diag_share = np.trace(od_by_name.values) / od_by_name.values.sum()

print("Filtered rows:", len(cz_kk))
print("Duplicated origin-destination pairs:", dup_pairs)
print("OD matrix shape:", od_by_name.shape)
print("Cells filled with 0 (missing pairs):", n_missing_cells)
print("Same names on rows and columns:", same_names)
print("Total commuters (all cells):", int(od_by_name.values.sum()))
if same_names:
    print("Within-kraj share (diagonal):", round(diag_share, 3))

# Check: the filtered rows must contain a single activity type and a single indicator
print("Activity values in filtered rows:", cz_kk[COL_ACTIVITY].unique().tolist())
print("Indicator values in filtered rows:", cz_kk[COL_INDICATOR].unique().tolist())

In [ ]:
# Local config: GISCO property names and output file name
GISCO_COL_ID, GISCO_COL_NAME, GISCO_COL_LEVEL = "NUTS_ID", "NUTS_NAME", "LEVL_CODE"
NUTS_LEVEL = 3
OD_CSV = CHECKPOINT_DIR / "od_observed.csv"

# Read only the feature properties (raw JSON avoids the geometry problem)
with open(GISCO_GEOJSON, encoding="utf-8") as f:
    gisco_props = pd.DataFrame([ft["properties"] for ft in json.load(f)["features"]])

gisco_country = gisco_props[
    gisco_props[GISCO_COL_ID].str.startswith(COUNTRY)
    & (gisco_props[GISCO_COL_LEVEL] == NUTS_LEVEL)
]
name_to_code = dict(zip(gisco_country[GISCO_COL_NAME], gisco_country[GISCO_COL_ID]))

# Every kraj name from the census must have a code
unmatched = sorted(set(od_by_name.index) - set(name_to_code))

od_coded = od_by_name.rename(index=name_to_code, columns=name_to_code)

# Same region set and same order as the cz table
ids = cz[COL_ID].tolist()
same_region_set = set(od_coded.index) == set(ids) == set(od_coded.columns)
od_coded = od_coded.loc[ids, ids]

assert not unmatched, "unmatched kraj names: " + ", ".join(unmatched)
assert same_region_set, "region codes differ from the loaded regions"
assert list(od_coded.index) == ids and list(od_coded.columns) == ids

total_before = int(od_by_name.values.sum())
total_after = int(od_coded.values.sum())

od_coded.to_csv(OD_CSV)

print("Unmatched names:", len(unmatched))
print("Same region set as cz:", same_region_set)
print("OD matrix shape:", od_coded.shape)
print("Total commuters before / after rename:", total_before, "/", total_after)
print("Saved:", os.path.relpath(OD_CSV))

In [ ]:
# Local config: skmob column names
TILE_ID = "tile_ID"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"

# Tessellation: one row per region (id, geometry, population)
tessellation = cz[[COL_ID, "geometry", COL_POP]].rename(columns={COL_ID: TILE_ID})
tessellation = gpd.GeoDataFrame(tessellation, geometry="geometry", crs=CRS_GEO)

# Flows in long format: one row per origin-destination pair (diagonal included).
# Axis names are set explicitly because rows and columns carried the same name.
flows_long = (
    od_coded
    .rename_axis(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST)
    .stack()
    .rename(COL_FLOW)
    .reset_index()
)

fdf = skmob.FlowDataFrame(
    flows_long,
    origin=COL_FLOW_ORIGIN,
    destination=COL_FLOW_DEST,
    flow=COL_FLOW,
    tessellation=tessellation,
    tile_id=TILE_ID,
)

print("Tiles in tessellation:", len(tessellation))
print("Flow rows:", len(fdf), "| expected:", n_regions ** 2)
print("Total commuters in fdf:", int(fdf[COL_FLOW].sum()), "| OD matrix:", int(od_coded.values.sum()))

In [ ]:
# Local config
OUTFLOW_COL = "tot_outflows"
TESSELLATION_CSV = CHECKPOINT_DIR / "tessellation_outflows.csv"

# Observed matrix without the diagonal (plain numpy array, no DataFrame view tricks)
observed_offdiag = od_coded.to_numpy(dtype=float).copy()
np.fill_diagonal(observed_offdiag, 0)

# Row sums = number of commuters leaving each region for another region
tot_outflows = pd.Series(observed_offdiag.sum(axis=1), index=od_coded.index)
tessellation[OUTFLOW_COL] = tot_outflows.loc[tessellation[TILE_ID]].to_numpy()

# Checks
total_all = float(od_coded.to_numpy().sum())
total_diag = float(np.trace(od_coded.to_numpy(dtype=float)))
total_off = float(observed_offdiag.sum())
assert np.isclose(total_off, total_all - total_diag)
assert (tessellation[OUTFLOW_COL] > 0).all(), "a region has zero outflow"

tessellation[[TILE_ID, COL_POP, OUTFLOW_COL]].to_csv(TESSELLATION_CSV, index=False)

print("Off-diagonal total:", int(total_off))
print("Total - diagonal:", int(total_all - total_diag))
print("Smallest outflow:", int(tessellation[OUTFLOW_COL].min()))
print("Saved:", os.path.relpath(TESSELLATION_CSV))

In [ ]:
# Local config
GRAVITY_TYPE = "singly constrained"
GRAVITY_VARIANTS = {"gravity_power": "power_law", "gravity_exp": "exponential"}
GRAVITY_PARAMS_CSV = CHECKPOINT_DIR / "gravity_params.csv"

gravity_models = {}
param_rows = []

for name, deterrence in GRAVITY_VARIANTS.items():
    # Record warnings instead of silencing them; only the class names are kept
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = Gravity(gravity_type=GRAVITY_TYPE, deterrence_func_type=deterrence)
        model.fit(fdf, relevance_column=COL_POP)

    gravity_models[name] = model
    param_rows.append({
        "model": name,
        "deterrence_func_type": model.deterrence_func_type,
        "gamma": model.deterrence_func_args[0],
        "origin_exp": model.origin_exp,
        "destination_exp": model.destination_exp,
        "gravity_type": model.gravity_type,
        "fit_warnings": ";".join(sorted({w.category.__name__ for w in caught})),
    })

gravity_params = pd.DataFrame(param_rows)
gravity_params.to_csv(GRAVITY_PARAMS_CSV, index=False)

print(gravity_params.to_string(index=False))
print("Saved:", os.path.relpath(GRAVITY_PARAMS_CSV))

In [ ]:
# Local config
MODEL_OBJECTS = {**gravity_models, "radiation": Radiation()}
REALIZATION_PATH = lambda name: CHECKPOINT_DIR / f"realizations_{name}.npy"

realizations = {}

for name in MODELS:
    path = REALIZATION_PATH(name)

    if path.exists():
        realizations[name] = np.load(path)
        print(name, "- loaded from checkpoint:", os.path.relpath(path))
        continue

    model = MODEL_OBJECTS[name]
    mats = []
    warning_types = set()
    for i in range(N_REPS):
        np.random.seed(SEED_BASE + i)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            synth = model.generate(
                tessellation,
                tile_id_column=TILE_ID,
                tot_outflows_column=OUTFLOW_COL,
                relevance_column=COL_POP,
                out_format="flows",
            )
        warning_types |= {w.category.__name__ for w in caught}

        # FlowDataFrame -> plain DataFrame -> matrix in the region order of cz
        mat = (
            pd.DataFrame(synth)
            .pivot(index=COL_FLOW_ORIGIN, columns=COL_FLOW_DEST, values=COL_FLOW)
            .reindex(index=ids, columns=ids)
            .fillna(0)
            .to_numpy()
        )
        mats.append(mat)

    realizations[name] = np.stack(mats)          # shape: (N_REPS, n_regions, n_regions)
    np.save(path, realizations[name])
    print(name, "- generated and saved:", os.path.relpath(path),
          "| warnings:", ";".join(sorted(warning_types)) or "none")

print()
for name in MODELS:
    arr = realizations[name]
    diag_sum = float(np.trace(arr, axis1=1, axis2=2).sum())
    mean_total = float(arr.sum(axis=(1, 2)).mean())
    print(f"{name}: shape {arr.shape} | diagonal sum {diag_sum:.0f} | "
          f"mean total {mean_total:.0f} (observed off-diagonal {observed_offdiag.sum():.0f})")

In [ ]:
# Local config
METRICS_CSV = CHECKPOINT_DIR / "metrics_per_realization.csv"
SUMMARY_CSV = CHECKPOINT_DIR / "metrics_summary.csv"


def cpc(observed, predicted):
    """Common Part of Commuters: 2 * sum(min) / (sum(obs) + sum(pred))."""
    return 2 * np.minimum(observed, predicted).sum() / (observed.sum() + predicted.sum())


def r2_positive(observed, predicted):
    """R2 on the raw scale, only on pairs with observed flow > 0."""
    mask = observed > 0
    return r2_score(observed[mask], predicted[mask])


# Region indices without the capital (position in the id order of cz)
keep = np.array([i for i, c in enumerate(ids) if c != CAPITAL_ID])
no_cap = np.ix_(keep, keep)
assert len(keep) == n_regions - 1

rows = []
for name in MODELS:
    for rep, pred in enumerate(realizations[name]):
        rows.append({
            "model": name,
            "rep": rep,
            "cpc": cpc(observed_offdiag, pred),
            "r2": r2_positive(observed_offdiag, pred),
            "cpc_no_capital": cpc(observed_offdiag[no_cap], pred[no_cap]),
        })

metrics = pd.DataFrame(rows)
metrics.to_csv(METRICS_CSV, index=False)

metric_cols = ["cpc", "r2", "cpc_no_capital"]
summary = metrics.groupby("model", sort=False)[metric_cols].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
summary.to_csv(SUMMARY_CSV, index=False)

print(summary.round(4).to_string(index=False))
print("Saved:", os.path.relpath(METRICS_CSV), "and", os.path.relpath(SUMMARY_CSV))

In [ ]:
import powerlaw

# Local config
POWERLAW_CSV = CHECKPOINT_DIR / "powerlaw_distance_test.csv"

# One data point per commuter: distance of the pair, repeated by the flow (diagonal excluded)
pair_mask = ~np.eye(n_regions, dtype=bool)
pair_dist = dist_matrix_km[pair_mask]
pair_flow = observed_offdiag[pair_mask].astype(int)
distances_weighted = np.repeat(pair_dist, pair_flow)

# Fit and compare; warnings are recorded (class names only) so that no file path is printed
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fit = powerlaw.Fit(distances_weighted, discrete=False)
    R, p = fit.distribution_compare("power_law", "exponential")

result = pd.DataFrame([{
    "n_points": len(distances_weighted),
    "n_distinct_distances": int(np.unique(pair_dist.round(6)).size),
    "xmin": fit.power_law.xmin,
    "power_law_alpha": fit.power_law.alpha,
    "R_power_vs_exp": R,
    "p": p,
    "warnings": ";".join(sorted({w.category.__name__ for w in caught})) or "none",
}])
result.to_csv(POWERLAW_CSV, index=False)

print(result.round(4).T.to_string(header=False))
print("Saved:", os.path.relpath(POWERLAW_CSV))

In [ ]:
# Local config
SCATTER_FILE = "cz_observed_vs_predicted.png"
FIG_DPI = 150
LABEL_X, LABEL_Y = "Megfigyelt flow", "Becsült flow"

obs_flat = observed_offdiag.flatten()

fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 5), sharex=True, sharey=True)
excluded = {}

for ax, name in zip(axes, MODELS):
    pred_flat = realizations[name].mean(axis=0).flatten()   # mean of the realizations
    on_observed = obs_flat > 0
    plotted = on_observed & (pred_flat > 0)
    excluded[name] = int(on_observed.sum() - plotted.sum())

    ax.scatter(obs_flat[plotted], pred_flat[plotted], alpha=0.6, s=20)
    lims = [min(obs_flat[plotted].min(), pred_flat[plotted].min()),
            max(obs_flat[plotted].max(), pred_flat[plotted].max())]
    ax.plot(lims, lims, "k--", linewidth=1)       # y = x reference line
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(LABEL_X)
    ax.set_title(name)

axes[0].set_ylabel(LABEL_Y)
plt.tight_layout()

for fig_dir in FIG_DIRS:
    out_path = fig_dir / SCATTER_FILE
    fig.savefig(out_path, dpi=FIG_DPI)
    print("Saved:", os.path.relpath(out_path))
plt.show()

print("Observed pairs left out (predicted flow = 0):", excluded)

In [ ]:
import matplotlib.lines as mlines
import matplotlib.patheffects as pe

# Local config
MAP_FILE_NAMES = "cz_commuting_network.png"
MAP_FILE_CODES = "cz_commuting_network_codes.png"
EDGE_WIDTH_MIN, EDGE_WIDTH_MAX = 1.0, 14.0     # line width range (points)
EDGE_ALPHA, EDGE_COLOR = 0.55, "steelblue"
NODE_SIZE_DIVISOR = 3500                       # marker area = population / divisor
NODE_COLOR = "darkred"
LABEL_OFFSET_BASE_M, LABEL_OFFSET_PER_SQRT_M = 10000, 500   # label shift from node (metres)
LABEL_BELOW_IDS = ["CZ020"]                    # labels placed below the node (avoids overlap with the capital)
MAP_TITLE = "CZ NUTS3 ingázási hálózat (valódi flow, 2021)"

code_to_name = {v: k for k, v in name_to_code.items()}
max_flow = observed_offdiag.max()
node_sizes = cz[COL_POP].to_numpy() / NODE_SIZE_DIVISOR


def draw_flow_map(label_by_id, file_name, title):
    """Draw the observed flow network; label_by_id maps region id -> label text."""
    fig, ax = plt.subplots(figsize=(12, 10))
    cz_metric.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=1)

    # Edges: width scaled by the square root of the flow
    n_edges = 0
    for i in range(n_regions):
        for j in range(n_regions):
            flow = observed_offdiag[i, j]
            if flow > 0:
                lw = EDGE_WIDTH_MIN + (EDGE_WIDTH_MAX - EDGE_WIDTH_MIN) * np.sqrt(flow / max_flow)
                ax.plot([coords[i, 0], coords[j, 0]], [coords[i, 1], coords[j, 1]],
                        linewidth=lw, alpha=EDGE_ALPHA, color=EDGE_COLOR,
                        zorder=1, solid_capstyle="round")
                n_edges += 1

    # Nodes: area scaled by population
    ax.scatter(coords[:, 0], coords[:, 1], s=node_sizes, color=NODE_COLOR,
               zorder=5, edgecolor="white", linewidth=1.2)

    # Labels: above the node, or below it for the ids in LABEL_BELOW_IDS
    for i, region_id in enumerate(ids):
        shift = np.sqrt(node_sizes[i]) * LABEL_OFFSET_PER_SQRT_M + LABEL_OFFSET_BASE_M
        below = region_id in LABEL_BELOW_IDS
        ax.annotate(label_by_id[region_id],
                    (coords[i, 0], coords[i, 1] + (-shift if below else shift)),
                    fontsize=9, fontweight="bold", ha="center",
                    va="top" if below else "bottom", zorder=6,
                    path_effects=[pe.withStroke(linewidth=3, foreground="white")])

    ax.set_title(title, fontsize=13)
    ax.set_axis_off()
    ax.legend(handles=[
        mlines.Line2D([], [], color=EDGE_COLOR, linewidth=6, alpha=EDGE_ALPHA,
                      label="Él vastagsága ∝ √(ingázók száma)"),
        mlines.Line2D([], [], marker="o", linestyle="", color=NODE_COLOR, markersize=10,
                      label="Csomópont mérete ∝ népesség"),
    ], loc="lower left", fontsize=10, frameon=True, framealpha=0.9)

    plt.tight_layout()
    for fig_dir in FIG_DIRS:
        out_path = fig_dir / file_name
        fig.savefig(out_path, dpi=FIG_DPI)
        print("Saved:", os.path.relpath(out_path))
    plt.show()
    return n_edges


# Figure 1: labels = Czech kraj names from the GISCO file
n_edges = draw_flow_map({c: code_to_name[c] for c in ids}, MAP_FILE_NAMES, MAP_TITLE)
print("Edges drawn:", n_edges)
print("Largest flow:", int(max_flow))

In [ ]:
# Figure 2: labels = NUTS3 codes
n_edges_codes = draw_flow_map({c: c for c in ids}, MAP_FILE_CODES, MAP_TITLE + " – NUTS3 kódok")
print("Edges drawn:", n_edges_codes)

In [ ]:
# Local config: edge attribute names (predicted_<model>), node attribute name
ATTR_POP, ATTR_WEIGHT = "population", "weight"
mean_pred = {name: realizations[name].mean(axis=0) for name in MODELS}

G_cz = nx.DiGraph()
for i, node_id in enumerate(ids):
    G_cz.add_node(node_id, **{ATTR_POP: int(cz[COL_POP].iloc[i])})

for i, origin in enumerate(ids):
    for j, dest in enumerate(ids):
        flow = observed_offdiag[i, j]
        if flow > 0:
            attrs = {ATTR_WEIGHT: float(flow)}
            for name in MODELS:
                attrs[f"predicted_{name}"] = float(mean_pred[name][i, j])
            G_cz.add_edge(origin, dest, **attrs)

nx.write_graphml(G_cz, GRAPH_PATH)

# Read back and check
G_check = nx.read_graphml(GRAPH_PATH)
assert G_check.number_of_nodes() == G_cz.number_of_nodes()
assert G_check.number_of_edges() == G_cz.number_of_edges()
total_weight = sum(d[ATTR_WEIGHT] for _, _, d in G_check.edges(data=True))

print("Nodes:", G_cz.number_of_nodes(), "| edges:", G_cz.number_of_edges())
print("Total edge weight:", int(total_weight), "| observed off-diagonal:", int(observed_offdiag.sum()))
print("Edge attributes:", sorted(next(iter(G_check.edges(data=True)))[2].keys()))
print("Saved:", os.path.relpath(GRAPH_PATH))